# 02 - CafeBERT base model: 5-fold OOF + full-train refit (seed=1337)

Outputs `artifacts/probs/cafebert_{oof,val,test}.npy` and `artifacts/metrics/cafebert.json`.

Different seed from PhoBERT (1337) per the plan, to add diversity to the ensemble.

In [ ]:
%pip install -q transformers datasets accelerate scikit-learn pandas numpy sentencepiece

In [ ]:
import sys, os

REPO_ROOT = '/content/drive/MyDrive/thesis/topicmodeling'
TMP_ROOT = '/content/ensemble_tmp'

if 'google.colab' in sys.modules:
    from google.colab import drive
    if not os.path.ismount('/content/drive'):
        drive.mount('/content/drive')
    if REPO_ROOT not in sys.path:
        sys.path.insert(0, REPO_ROOT)
else:
    import pathlib
    LOCAL = pathlib.Path.cwd().resolve()
    while LOCAL.name != 'tm_research' and LOCAL.parent != LOCAL:
        LOCAL = LOCAL.parent
    sys.path.insert(0, str(LOCAL.parent))

from tm_research.ensemble.colab_setup import setup_colab
paths = setup_colab(repo_root=REPO_ROOT, tmp_root=TMP_ROOT)

In [ ]:
from tm_research.ensemble.utils_io import load_splits
from tm_research.ensemble.bert_oof import BertOOFConfig, run_bert_oof

train_df, val_df, test_df, label_map = load_splits()
print('train', train_df.shape, 'val', val_df.shape, 'test', test_df.shape)
print('classes', label_map.class_names)

In [ ]:
cfg = BertOOFConfig(
    model_name='uitnlp/CafeBERT',
    output_name='cafebert',
    seed=1337,
    n_folds=5,
    num_epochs=6,
    learning_rate=2e-5,
    max_length=256,
    train_batch_size=16,
    eval_batch_size=32,
    work_dir=str(paths.bert_work / 'cafebert'),
)
metrics = run_bert_oof(cfg, train_df, val_df, test_df, label_map)
metrics